# 11 · Error analysis and shortcut diagnostics
**Purpose:** explain *why* detectors succeed/fail: length, generator, prompt template, script, calibration, error overlap between detectors, interpretable n-gram features (shortcut check), and a qualitative sample of errors for native-speaker reading.

**Outputs:** `08_results/error_*.csv`, `08_results/qualitative_errors_{lang}.csv`. **Runtime:** CPU.

In [ ]:
# ==== Bootstrap (identical in every notebook) ====
NOTEBOOK = "11_error_analysis"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
import os, json
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()
log = get_logger(NOTEBOOK); _ = env_snapshot(NOTEBOOK)

In [ ]:
ds = load_dataset(); reg = load_registry(); reg = reg[reg.status == "done"]
test = ds[ds.split == "test"].copy(); val = ds[ds.split == "val"].copy()
def main_runs():
    r = {d: reg[(reg.exp == "E2") & (reg.train == "all") & (reg.heldout == "none") & (reg.detector == d)].run_id.tolist()
         for d in ("mdeberta", "xlmr", "ngram", "stylo")}
    r |= {d: reg[(reg.exp == "E5") & (reg.detector == d)].run_id.tolist() for d in ("binoculars-generic", "binoculars-kazakh", "fastdetect-generic", "fastdetect-kazakh")}
    return {k: v for k, v in r.items() if v}
MAIN = main_runs()
def mean_score(rids, ids):
    return np.mean([pd.read_parquet(pred_path(r)).set_index("doc_id").score.reindex(ids).values for r in rids], axis=0)
for det, rids in MAIN.items():
    test[det] = mean_score(rids, test.doc_id); val[det] = mean_score(rids, val.doc_id)
log.info(f"main detectors: {list(MAIN)}")

## 1 · Length: AUROC within length tertiles (removes length as an explanation)

In [ ]:
test["len_bin"] = pd.qcut(test.n_words, 3, labels=["short", "mid", "long"])
rows = []
for det in MAIN:
    for (l, b), g in test.groupby(["lang", "len_bin"], observed=True):
        if g.label.nunique() == 2:
            rows.append({"detector": det, "lang": l, "len_bin": b, "n": len(g), "auroc": auroc(g.label, g[det])})
    rows.append({"detector": det, "lang": "all", "len_bin": "length-only baseline", "n": len(test), "auroc": auroc(test.label, test.n_words)})
err_len = pd.DataFrame(rows); err_len.to_csv(P["results"] / "error_length.csv", index=False)
err_len.pivot_table(index=["detector", "len_bin"], columns="lang", values="auroc").round(3)

## 2 · Generator and prompt-template effects (machine texts vs. all humans of the language)

In [ ]:
rows = []
for det in MAIN:
    for l, g in test.groupby("lang"):
        h = g[g.label == 0]
        for key in ("generator", "prompt_id"):
            for v, m in g[g.label == 1].groupby(key):
                d = pd.concat([h, m]); rows.append({"detector": det, "lang": l, "factor": key, "level": v, "auroc": auroc(d.label, d[det])})
err_gen = pd.DataFrame(rows); err_gen.to_csv(P["results"] / "error_generator_prompt.csv", index=False)
err_gen[err_gen.factor == "generator"].pivot_table(index=["detector", "level"], columns="lang", values="auroc").round(3)

## 3 · Error overlap between detectors (Jaccard of misclassified test docs at the validation-F1 threshold)

In [ ]:
wrong = {}
for det in MAIN:
    thr = best_f1_threshold(val.label.values, val[det].values)
    wrong[det] = set(test.doc_id[(test[det] > thr).astype(int) != test.label])
J = pd.DataFrame([[len(wrong[a] & wrong[b]) / max(1, len(wrong[a] | wrong[b])) for b in MAIN] for a in MAIN], index=list(MAIN), columns=list(MAIN))
J.to_csv(P["results"] / "error_overlap_jaccard.csv"); J.round(2)

## 4 · Interpretable shortcut check: top character n-grams of per-language n-gram models, per class

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
NG = CFG["ngram"]; tr = ds[ds.split == "train"]
rows = []
for l in LANGS:
    t = tr[tr.lang == l]
    vec = TfidfVectorizer(analyzer=NG["analyzer"], ngram_range=tuple(NG["ngram_range"]), max_features=NG["max_features"], sublinear_tf=True, min_df=2, lowercase=False)
    clf = LogisticRegression(C=1.0, max_iter=3000, solver="liblinear").fit(vec.fit_transform(t.text), t.label)
    names = np.array(vec.get_feature_names_out()); w = clf.coef_[0]; o = np.argsort(w)
    rows += [{"lang": l, "class": "machine", "rank": i + 1, "ngram": repr(names[j]), "weight": w[j]} for i, j in enumerate(o[::-1][:25])]
    rows += [{"lang": l, "class": "human", "rank": i + 1, "ngram": repr(names[j]), "weight": w[j]} for i, j in enumerate(o[:25])]
feat = pd.DataFrame(rows); feat.to_csv(P["results"] / "error_top_ngrams.csv", index=False)
feat[feat["rank"] <= 8].pivot_table(index=["class", "rank"], columns="lang", values="ngram", aggfunc="first")

## 5 · Qualitative sample for native-speaker review (most confident errors of the main fine-tuned model)

In [ ]:
det = "mdeberta" if "mdeberta" in MAIN else list(MAIN)[0]
thr = best_f1_threshold(val.label.values, val[det].values)
for l, g in test.groupby("lang"):
    fn = g[(g.label == 1) & (g[det] <= thr)].nsmallest(10, det).assign(error="FN (machine judged human)")
    fp = g[(g.label == 0) & (g[det] > thr)].nlargest(10, det).assign(error="FP (human judged machine)")
    pd.concat([fn, fp])[["doc_id", "lang", "error", "generator", "prompt_id", det, "n_words", "text"]].to_csv(
        P["results"] / f"qualitative_errors_{l}.csv", index=False)
log.info("qualitative error files written — ask native speakers to annotate a 'why' column (fluency, factual errors, style, topic)")

## 6 · Generation yield (LLM support for Turkic languages) — from notebook 03

In [ ]:
pd.read_csv(P["validated"] / "generation_yield.csv")